# Episode 27: Speculative decoding from scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep027-speculative-decoding.ipynb)

A draft guesses the next few tokens, and our agent checks them in the pass that writes its own, so it writes what greedy decoding writes in fewer passes. Two drafts, prompt lookup and a small model trained for thirty minutes, measured on the agent's tasks.

[The video](https://youtu.be/gyC0YdAOlio) · [The episode's page](https://mvgpt.nenuadrian.com/ep027-speculative-decoding/) · [The docs](https://nenuadrian.github.io/mvGPT/speculative-decoding/) · [The code at `ep027/end`](https://github.com/nenuadrian/mvGPT/tree/ep027/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep027/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


import shlex
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request


def start(command, url):
    # A server in the background, once it answers at url; its output goes to /tmp/server-<port>.log.
    log = open(f"/tmp/server-{urllib.parse.urlparse(url).port}.log", "w")
    server = subprocess.Popen(shlex.split(command), stdout=log, stderr=subprocess.STDOUT)
    while True:
        try:
            urllib.request.urlopen(url, timeout=1)
            return server
        except urllib.error.HTTPError:
            return server
        except OSError:
            if server.poll() is not None:
                raise RuntimeError(open(log.name).read())
            time.sleep(1)


def stop(server):
    server.terminate()
    server.wait()


fetch("graph-agent")

## smollerGraph
The graph the agent asks, as in episode 25, with `sg serve` on port 8474, where mvGPT looks
for it.

In [ ]:
if not os.path.isdir("/content/smollerGraph"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep010/end https://github.com/nenuadrian/smollerGraph /content/smollerGraph
!pip install --quiet cmake ninja
!cmake -S /content/smollerGraph -B /content/smollerGraph/build -G Ninja > /dev/null && cmake --build /content/smollerGraph/build --target sg -- --quiet
!rm -f /content/smollerGraph/wikidata.sg && /content/smollerGraph/build/sg load /content/smollerGraph/wikidata.sg /content/smollerGraph/data/wikidata.txt
sg = start("/content/smollerGraph/build/sg serve /content/smollerGraph/wikidata.sg 8474", "http://127.0.0.1:8474/api/names?prefix=R")

## A pass, by how many tokens it adds
Episode 25's agent, from the Hub: a forward pass after a context in the KV cache, adding a
token and adding several, on the GPU and on the CPU.

In [ ]:
!python speculative_bench.py checkpoints/graph-agent/model.pt

## The tests
The same text as greedy decoding, each guess kept when the draft is the model, lookup's guesses, and the cache trimmed.

In [ ]:
!python -m pytest -q tests/test_speculative.py

## Guesses from the text
The graph role's query for the King Penguin, with lookup guessing from the tokens so far.

In [ ]:
!python speculative.py checkpoints/graph-agent/model.pt "How much does the King Penguin weigh?" --draft lookup

## The data
Episode 21's tasks (they need FineWeb-Edu's last file for the web pages), episode 24's
questions and episode 25's tasks, which the draft model trains on.

In [ ]:
!python -c "from data import WEB_SHARDS, download_shard; download_shard(WEB_SHARDS - 1)"  # FineWeb-Edu's last file, which pretraining never read
!python agent_data.py
!python graph_data.py /content/smollerGraph/data/wikidata.txt
!python graph_agent_data.py /content/smollerGraph/data/wikidata.txt

## A draft model
A GPT of four layers, about a fifth of the agent, trained from scratch for thirty minutes on
episode 25's training set, to `checkpoints/draft`. The episode's run on the Mac took 7,954
steps.

In [ ]:
!python draft_sft.py 30

## Guesses from the draft
The King Penguin's query again, then the agent with the draft guessing for each of its contexts.

In [ ]:
!python speculative.py checkpoints/graph-agent/model.pt "How much does the King Penguin weigh?" --draft checkpoints/draft/model.pt
!python agent.py checkpoints/graph-agent/model.pt "How much heavier is the Okapi than the Lion?" --names task --draft checkpoints/draft/model.pt

## Measuring it
Episode 25's tasks with no draft, with lookup and with the draft model: right, the same text, and the time.

In [ ]:
!python speculative_eval.py checkpoints/graph-agent/model.pt checkpoints/draft/model.pt

Next: [episode 28, Attention heads in our GPT, seen and tested](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep028-attention.ipynb)